# Lab: Bring your own custom container with Amazon SageMaker

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.0.3</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>


## Overview

### Background
Here, we'll show how to bring your docker cotainer that packages your environment and code. We showcase the [decision tree](http://scikit-learn.org/stable/modules/tree.html) algorithm from the widely used [scikit-learn](http://scikit-learn.org/stable/) machine learning package. The example is purposefully fairly trivial since the point is to show the surrounding structure that you'll want to add to your own container so you can bring it to Amazon SageMaker for training and hosting.


### High-level overview

The following diagram shows how you typically train and deploy a model with Amazon SageMaker:

<div>
<img src="https://docs.aws.amazon.com/sagemaker/latest/dg/images/sagemaker-architecture.png" width="900"/>
</div>

The area labeled SageMaker highlights the two components of SageMaker: model training and model deployment. The area labeled [EC2 container registry](https://aws.amazon.com/ecr/) is where we store, manage, and deploy our Docker container images. The training data and model artifacts are stored in S3 bucket. 

In this lab, we use a single image to support both model training and hosting for simplicity. Sometimes you’ll want separate images for training and hosting because they have different requirements. 

The high-level steps include:
1. **Building the container** - We walk through the different components of the containers and inspect the docker file. Then we build and push the container to ECR. 
2. **Setup & Upload Data** - Once our container is built and registered. We ready sagemaker and upload the data to S3. 
3. **Model Training** - Create a training job using SageMaker Python SDK. It will pull data from S3 and use the container we built.  
4. **Model Deployment** - Once training is complete, deploy our model to a HTTP endpoint using SageMaker Python SDK. 
5. **Run Inferences** - Run predictions to test our model.
6. **Cleanup**



## Building the container
[Docker](https://aws.amazon.com/docker/#:~:text=Docker%20is%20a%20software%20platform,test%2C%20and%20deploy%20applications%20quickly.&text=Running%20Docker%20on%20AWS%20provides,distributed%20applications%20at%20any%20scale.) packages software into standardized units called [containers](https://aws.amazon.com/containers/) that have everything the software needs to run including libraries, system tools, code, and runtime. Using Docker, you can quickly deploy and scale applications into any environment and know your code will run.


Amazon SageMaker uses Docker to allow users to train and deploy arbitrary algorithms. More details on [how to use docker containers with sagemaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html).

### Walkthrough of the container directory
You can find the source code of the sample container we are using in [this GitHub repository](https://github.com/aws/amazon-sagemaker-examples/tree/main/advanced_functionality/scikit_bring_your_own). 

The container directory contains all the components you need to package for SageMaker:

```
.
|-- Dockerfile
`-- decision_trees
    |-- nginx.conf
    |-- predictor.py
    |-- serve
    |-- train
    `-- wsgi.py
```

Let’s discuss each of these in turn:

- `Dockerfile` describes how to build your Docker container image. More details below. 
- `decision_trees` is the directory which contains the files that will be installed in the container.

In this simple application, we only install five files in the container. These five show the standard structure of our Python containers, although you are free to choose a different toolset or programming language and therefore could have a different layout.

The files that we’ll put in the container are:

- `nginx.conf` is the configuration file for the nginx front-end. Generally, you should be able to take this file as-is.
- `predictor.py` is the program that actually implements the Flask web server and the decision tree predictions for this app. You’ll want to customize the actual prediction parts to your application. Since this algorithm is simple, we do all the processing here in this file, but you may choose to have separate files for implementing your custom logic.
- `serve` is the program started when the container is started for hosting. It simply launches the gunicorn server which runs multiple instances of the Flask app defined in predictor.py. You should be able to take this file as-is.
- `train` is the program that is invoked when the container is run for training. You will modify this program to implement your training algorithm.
- `wsgi.py` is a small wrapper used to invoke the Flask app. You should be able to take this file as-is.

In summary, the two files you will probably want to change for your application are `train` and `predictor.py`

### Install packages
Please choose `Python 3 (ipykernel)` kernel to proceed.

We will first install the prerequisite packages. They should be already installed if you run the [Setup.ipynb](../Setup.ipynb) notebook. If you experience problems, uncomment the following three cells to re-install the right libraires and the restart the kernel and then continue

In [ ]:
# --- Shared dependencies (managed via uv) ------------------------------------
# Installs the shared kernel dependencies from the repo-root requirements.txt
# into THIS notebook's kernel using uv. Idempotent and fast when already
# satisfied. Full first-run setup (SageMaker SDK suite + kernel restart) lives
# in the top-level Setup.ipynb.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r ../../requirements.txt

In [ ]:
# # Restart kernel to get the packages
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

### Install Docker

To use Docker, you must manually install it from the terminal of your JupyterLab application. Please get familiar with the docker operations that are currently supported in Studio [see here](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html).


In [ ]:
%%bash

# see https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository
sudo apt-get update
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

# pick the latest patch from:
sudo apt-get install docker-ce-cli -y

# IMPORTANT: do NOT install the 'docker-compose-plugin' apt package on SageMaker Studio.
# On Studio it installs Docker Compose v5.x, which issues Docker network management API
# calls during 'docker compose up'. The Studio Docker proxy (unix:///docker/proxy.sock)
# forbids those calls, so local mode fails with:
#   'The request for current resource is not allowed on SageMaker Studio.'
# Compose v2.x does not make those calls and works with the Studio proxy, so we pin it and
# install it as the docker CLI plugin instead.
DOCKER_COMPOSE_VERSION=v2.39.4
sudo mkdir -p /usr/libexec/docker/cli-plugins
sudo curl -fsSL https://github.com/docker/compose/releases/download/${DOCKER_COMPOSE_VERSION}/docker-compose-linux-$(uname -m) -o /usr/libexec/docker/cli-plugins/docker-compose
sudo chmod +x /usr/libexec/docker/cli-plugins/docker-compose

# validate the Docker Client is able to access Docker Server at [unix:///docker/proxy.sock]
docker version
# validate the Docker Compose plugin installed
docker compose version

# Install Compose Switch. this is a replacement to the Compose V1 docker-compose (python) executable.
# It translates the command line into Compose V2 docker compose then run the latter.
# this is needed by the sagemaker python sdk to run in local mode
# see https://github.com/docker/compose-switch
curl -fL https://raw.githubusercontent.com/docker/compose-switch/master/install_on_linux.sh | sudo sh
docker-compose version

### The Dockerfile
The `Dockerfile` describes the image that we want to build. You can think of it as describing the complete operating system installation of the system that you want to run. A Docker container running is quite a bit lighter than a full operating system, however, because it takes advantage of Linux on the host machine for the basic operations.

For the Python science stack, we will start from a standard Ubuntu installation and run the normal tools to install the things needed by `scikit-learn`. Finally, we add the code that implements our specific algorithm to the container and set up the right environment to run under.

Let's take a look of what's inside our `Dockerfile`:

In [ ]:
!pygmentize container/Dockerfile

### Building and registering the container

In [ ]:
%%sh
# Login to ECR
aws --region ${AWS_DEFAULT_REGION} ecr get-login-password | docker login --username AWS --password-stdin ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "sagemaker-decision-trees" > /dev/null 2>&1

if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "sagemaker-decision-trees" > /dev/null
fi

cd container

chmod +x decision_trees/train
chmod +x decision_trees/serve

# Build the image - it might take a few minutes to complete this step
docker build --network sagemaker . -t ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees:latest
# Push the image to ECR
docker push ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees:latest


WARNING! Your credentials are stored unencrypted in '/home/sagemaker-user/.docker/config.json'.
Con

figure a credential helper to remove this warning. See
https://docs.docker.com/go/credential-store/


Login Succeeded


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Bu

ildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-va

...

c0a556143f14: Pushed


222a5c67b449: Pushed


f103cd120fdd: Pushed


f96725a383c6: Pushed


latest: digest: sha256:fe8ae01b5d502f07cda6b1a9ddd856b6836a68329a0e684f1f829737a4a673e0 size: 1369


## Setup & Upload Data

### Setup the Environment 
Here we specify a bucket to use and the role that will be used for working with SageMaker.



In [ ]:
S3_prefix = "DEMO-scikit-byo-iris"

# Define IAM role
import boto3
import re

import os
import numpy as np
import pandas as pd
from sagemaker.core.helper.session_helper import get_execution_role

role = get_execution_role()

The session remembers our connection parameters to SageMaker. We’ll use it to perform all of our SageMaker operations.

In [ ]:
from sagemaker.core.helper.session_helper import Session
from time import gmtime, strftime

sess = Session()

### Upload data to S3 Bucket

When training large models with huge amounts of data, you’ll typically use big data tools, like Amazon Athena, AWS Glue, or Amazon EMR, to create your data in S3. For the purposes of this example, we’re using some the [classic Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set) in the `lab03_data` directory. 

We can use use the tools provided by the [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/) to upload the data to a default bucket.

In [ ]:
# cell 05

WORK_DIRECTORY = "data"

data_location = sess.upload_data(WORK_DIRECTORY, key_prefix=S3_prefix)

## Model Training

In SageMaker Python SDK V3, we use `ModelTrainer` to define how to use the container to train. This includes the configuration we need to invoke SageMaker training:

- `training_image (str)` - The [Amazon Elastic Container Registry](https://aws.amazon.com/ecr/) path where the docker image is registered.
- `role (str)` - SageMaker IAM role as obtained above.
- `compute (Compute)` - Compute configuration with `instance_type` and `instance_count`.
- `output_data_config (dict)` - Configuration with `s3_output_path` where the model artifact will be written.
- `sagemaker_session (Session)` - the SageMaker session object.

Then we use `model_trainer.train()` method to train against the data that we uploaded.
The API calls the Amazon SageMaker `CreateTrainingJob` API to start model training. Input data is specified using `InputData` objects with a `channel_name` and `data_source`.

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import Compute, InputData

account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name
image_uri = "{}.dkr.ecr.{}.amazonaws.com/sagemaker-decision-trees:latest".format(account, region)

tree = ModelTrainer(
    training_image=image_uri,
    role=role,
    compute=Compute(instance_type="ml.c5.2xlarge", instance_count=1),
    output_data_config={"s3_output_path": "s3://{}/output".format(sess.default_bucket())},
    sagemaker_session=sess,
    hyperparameters={
        "max_leaf_nodes": "2"
    },
)

file_location = data_location + "/iris.csv"
tree.train(input_data_config=[InputData(channel_name="training", data_source=file_location)], wait=True, logs=True)

[06/24/26 14:45:14] INFO     Base name not provided. Using default name:                             ]8;id=8523447;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8523448;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             sagemaker-decision-trees-job                                                          

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=8523454;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8523455;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=8523462;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8523463;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             381492082703.dkr.ecr.us-west-2.amazonaws.com/sagemaker-decision-t                     
                             rees:latest                                                                           

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8523470;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8523471;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


...

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [   =] Waiting for TrainingJob... 0:01:15                                                                       │
│ ⠦ Current status: InProgress                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [  ==] Waiting for TrainingJob... 0:01:15                                                                       │
│ ⠏ Current status: InProgress                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [=== ] Waiting for TrainingJob... 0:01:15                                                                       │
│ ⠹ Current status: InProgress                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [    ] Waiting for TrainingJob... 0:01:15                                                                       │
│ ⠴ Current status: InProgress                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/24/26 14:46:31] INFO     Final Resource Status: Completed                                    ]8;id=8523513;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8523514;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31442\31442]8;;\

## Model Deployment
You can use a trained model to get real time predictions using HTTP endpoint. Follow these steps to walk you through the process.

In SageMaker Python SDK V3, we use `ModelBuilder` to create a deployable model. After training completes, we:
1. Get the model artifacts location from `tree._latest_training_job.model_artifacts.s3_model_artifacts`
2. Create a `SchemaBuilder` with sample input/output for serialization
3. Create a `ModelBuilder` with the image URI, model data URL, and schema
4. Call `build()` to prepare the model, then `deploy()` to create the endpoint

The `deploy()` method returns an `Endpoint` object (not a Predictor like in V2). Use `endpoint.invoke()` to make predictions.

Key parameters:
- `image_uri (str)` – The container image for inference.
- `s3_model_data_url (str)` – S3 path to the model artifacts from training.
- `schema_builder (SchemaBuilder)` – Defines input/output serialization.
- `role_arn (str)` – IAM role for the endpoint.
- `mode (Mode)` – Optional. Use `Mode.LOCAL_CONTAINER` for local testing.

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder

# Get model data from training job
model_data_url = tree._latest_training_job.model_artifacts.s3_model_artifacts

schema_builder = SchemaBuilder(
    sample_input="5.1,3.5,1.4,0.2",
    sample_output="setosa",
)

model_builder = ModelBuilder(
    image_uri=image_uri,
    s3_model_data_url=model_data_url,
    schema_builder=schema_builder,
    role_arn=role,
)

model = model_builder.build()

endpoint = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
)


                    DEBUG    Auto-detecting optimal instance type for model...           ]8;id=8523521;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8523522;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=8523528;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8523529;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8523534;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8523535;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=8523541;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8523542;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1335\1335]8;;\
                             MLflow model input                                                                    

                    INFO     Creating model with name: model-129284da                        ]8;id=8523549;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=8523550;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1922\1922]8;;\

...

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [  ==] Waiting for Endpoint... 0:02:29                                                                          │
│ ⠴ Current status: Creating                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [  ==] Waiting for Endpoint... 0:02:29                                                                          │
│ ⠏ Current status: Creating                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [=== ] Waiting for Endpoint... 0:02:30                                                                          │
│ ⠹ Current status: Creating                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [    ] Waiting for Endpoint... 0:02:30                                                                          │
│ ⠴ Current status: Creating                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/24/26 14:49:04] INFO     ✅ Deployment successful: Endpoint 'endpoint-e1678b87' using     ]8;id=8523591;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=8523592;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2760\2760]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:us-west-2:381492082703:endpoint/endpoint-e1678                      
                             b87)                                                                                  

## Run Inferences


### Preparing test data
In order to do some predictions, we’ll extract some of the data we used for training and do predictions against it. This is, of course, bad statistical practice, but an easy way to see how the mechanism works.

In [ ]:
print(file_location)

s3://sagemaker-us-west-2-381492082703/DEMO-scikit-byo-iris/iris.csv


In [ ]:
import pandas as pd
shape = pd.read_csv(file_location, header=None)
shape.sample(3)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 import pandas as pd                                                                          │
│ ❱ 2 shape = pd.read_csv(file_location, header=None)                                              │
│   3 shape.sample(3)                                                                              │
│   4                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:948 in read_csv             │
│                                                                                                  │
│    945 │   )                                                                                     │
│    946 │   kwds.update(kwds_defaults)                                                            │
│    947 │                                                                                         │
│ ❱  948 │   return _read(filepath_or_buffer, kwds)                                                │
│    949                                                                                           │
│    950                                                                                           │
│    951 # iterator=True -> TextFileReader                                                         │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:611 in _read                │
│                                                                                                  │
│    608 │   _validate_names(kwds.get("names", None))                                              │
│    609 │                                                                                         │
│    610 │   # Create the parser.                                                                  │
│ ❱  611 │   parser = TextFileReader(filepath_or_buffer, **kwds)                                   │
│    612 │                                                                                         │
│    613 │   if chunksize or iterator:                                                             │
│    614 │   │   return parser                                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:1448 in __init__            │
│                                                                                                  │
│   1445 │   │   │   self.options["has_index_names"] = kwds["has_index_names"]                     │
│   1446 │   │                                                                                     │
│   1447 │   │   self.handles: IOHandles | None = None                                             │
│ ❱ 1448 │   │   self._engine = self._make_engine(f, self.engine)                                  │
│   1449 │                                                                                         │
│   1450 │   def close(self) -> None:                                                              │
│   1451 │   │   if self.handles is not None:                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:1705 in _make_engine        │
│                                                            

In [ ]:
# drop the label column in the training set
shape.drop(shape.columns[[0]], axis=1, inplace=True)
shape.sample(3)

In [ ]:
import itertools

a = [50 * i for i in range(3)]
b = [40 + i for i in range(10)]
indices = [i + j for i, j in itertools.product(a, b)]

test_data = shape.iloc[indices[:-1]]

### Predictions

In V3, the `deploy()` method returns an `Endpoint` object. Use `endpoint.invoke()` with `body` (the data) and `content_type` to make predictions. Alternatively, you can use the boto3 `sagemaker-runtime` client directly with `invoke_endpoint()`.

In [ ]:
import io
import pandas as pd

# If test_data is a DataFrame
csv_data = test_data.to_csv(header=False, index=False)
result = endpoint.invoke(body=csv_data, content_type="text/csv")
print(result.body.read().decode("utf-8"))


In [ ]:
import boto3

runtime = boto3.client("sagemaker-runtime")
response = runtime.invoke_endpoint(
    EndpointName=endpoint.endpoint_name,
    ContentType="text/csv",
    Body=test_data.to_csv(header=False, index=False)
)
print(response["Body"].read().decode("utf-8"))

## Cleanup
After completing the lab, use these steps to [delete the endpoint through AWS Console](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-cleanup.html) or simply run the following code


In [ ]:
sess.delete_endpoint(endpoint.endpoint_name)

Remove the container artifacts and data we downloaded.